In [0]:
%python
%pip install yfinance

In [0]:
%python
dbutils.widgets.text("process_datetime", "")
dbutils.widgets.text("runId", "")

process_datetime = dbutils.widgets.get("process_datetime")
runId = dbutils.widgets.get("runId")
written_records = 0

In [0]:
%python
import json

# Dependencias
import yfinance as yf
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
%python
# Listar todos los simbolos a cotizar con su fecha de la transacción
symbols_with_dates = [(row['simbolo_base'], row['fecha']) for row in spark.sql(f"SELECT DISTINCT simbolo_base, DATE(fecha) as fecha FROM silver.transacciones WHERE fecha_auditoria = '{process_datetime}' LIMIT 1000").collect()]

In [0]:
%python
# Descargar cotizaciones y guardar como dataframe
cotizaciones = []
for s, fecha in symbols_with_dates:
    try:
        # Descargar datos de Yahoo Finance
        data = yf.download(mapped, start=fecha)
        
        if data.empty:
            print('ok')
            # Si no hay datos, insertar registro con precio null
            cotizaciones.append({
                "simbolo_base": s,
                "fecha_cotizacion": fecha,
                "precio_mercado": None,
                "error_msg": "No data from Yahoo Finance"
            })
        else:
            print("ok")
            # Si hay datos, extraer el precio de cierre
            for idx, row in data.iterrows():
                # Extraer el valor numérico del Series
                precio = row['Close']
                if precio is not None:
                    # Convertir Series a valor escalar si es necesario
                    precio = float(precio.iloc[0]) if hasattr(precio, 'iloc') else float(precio)
                
                cotizaciones.append({
                    "simbolo_base": s,
                    "fecha_cotizacion": idx.strftime('%Y-%m-%d'),
                    "precio_mercado": precio,
                    "error_msg": None
                })
    except Exception as e:
        # Si falla la llamada a la API, insertar registro con precio null
        print('error')
        cotizaciones.append({
            "simbolo_base": s,
            "fecha_cotizacion": fecha,
            "precio_mercado": None,
            "error_msg": str(e)[:200]  # Limitar mensaje de error
        })

# Los datos están listos en la lista cotizaciones
# El conteo y guardado se maneja en las siguientes celdas

In [0]:
%python
records_read = len(cotizaciones)
if records_read == 0:
    result = {
        "status": "SUCCESS",
        "records_read": 0,
        "records_written": 0,
        "error_message": None,
        "table_name": "bronze.cotizaciones_mercado",
        "layer": "BRONZE"
    }
    dbutils.notebook.exit(json.dumps(result))

In [0]:
%python
result = {
    "status": "SUCCESS",
    "records_read": records_read,
    "records_written": written_records,
    "error_message": None,
    "table_name": "bronze.cotizaciones_mercado",
    "layer": "BRONZE"
}
dbutils.notebook.exit(json.dumps(result))